# 03 - BloodMNIST Tuning

Notebook này chỉ load feature `.npy` của BloodMNIST và chạy tuning cho warm-up và lambda sweep.


In [ ]:
import sys
import pandas as pd
import torch

sys.path.append('..')
from src.utils import get_device, load_feature_file, set_seed, train_clustering

set_seed(42)
device = get_device()
X, y = load_feature_file('../data/bloodmnist_resnet_features.npy', '../data/bloodmnist_labels.npy', device=device)
true_k = int(torch.unique(y).numel())
print('[INFO] BloodMNIST features:', tuple(X.shape), '| labels:', tuple(y.shape), '| K_true =', true_k)


In [ ]:
def run_cfg(name, k0=12, lambda_param=2.0, warmup_epochs=20, enable_split=True, enable_merge=True, epochs=50):
    k_star, acc, nmi, ari = train_clustering(
        features=X,
        labels=y,
        k_max=k0,
        device=device,
        epochs=epochs,
        lr=1e-3,
        lambda_param=lambda_param,
        gamma=0.1,
        warmup_epochs=warmup_epochs,
        enable_split=enable_split,
        enable_merge=enable_merge,
    )
    return {
        'Setting': name,
        'K0': k0,
        'lambda': lambda_param,
        'warmup_epochs': warmup_epochs,
        'K*': int(k_star),
        '|K*-K_true|': abs(int(k_star) - true_k),
        'ACC(%)': acc * 100,
        'NMI(%)': nmi * 100,
        'ARI(%)': ari * 100,
    }


In [ ]:
baseline_rows = [
    run_cfg('PnP dynamic', k0=12, lambda_param=2.0, warmup_epochs=20, enable_split=True, enable_merge=True),
    run_cfg('Fixed-K baseline', k0=12, lambda_param=2.0, warmup_epochs=20, enable_split=False, enable_merge=False),
]
pd.DataFrame(baseline_rows)


In [ ]:
lambda_rows = []
for lam in [1.5, 2.0, 2.5, 3.0]:
    lambda_rows.append(run_cfg(f'lambda={lam}', k0=12, lambda_param=lam, warmup_epochs=20, enable_split=True, enable_merge=True))

pd.DataFrame(lambda_rows)


In [ ]:
warmup_rows = []
for warmup in [0, 10, 20, 30]:
    warmup_rows.append(run_cfg(f'warmup={warmup}', k0=12, lambda_param=2.0, warmup_epochs=warmup, enable_split=True, enable_merge=True))

pd.DataFrame(warmup_rows)
